# L78 · 节拍追踪从零实现 — onset 包络、自相关（autocorrelation）与 BPM 估计

**学习目标**
- 理解 onset（音符起点）检测的原理：谱通量（Spectral Flux）
- 实现 onset_envelope（纯 NumPy + aurora.audio.stft）
- 用自相关估计节拍周期和 BPM
- 与 aurora.music.beat_track 对比验证

## 什么是 Onset？

**Onset** = 音符能量突然上升的时刻（鼓击、弦拨、音符开始）。

**谱通量（Spectral Flux）** = 相邻帧幅度谱的正差值之和：
```
flux[t] = Σ_k max(0, |S[t][k]| - |S[t-1][k]|)
```
大值 = 能量增加 = 可能是 onset。

**自相关估计 BPM**：如果节拍周期为 τ 帧，onset 包络每隔 τ 帧会出现一个峰值。
自相关在 lag=τ 处会出现最大值。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from aurora.audio.stft import stft

In [ ]:
# 生成测试信号：120 BPM 的鼓击模拟
SR = 16000
DURATION = 4.0
HOP = 256
N_FFT = 1024

t = np.linspace(0, DURATION, int(SR * DURATION), endpoint=False)

# 模拟鼓击：每 0.5 秒一个短暂脉冲
beat_period = SR // 2   # 120 BPM
signal = np.zeros(len(t))
for beat_start in range(0, len(t), beat_period):
    end = min(beat_start + SR // 50, len(t))   # 20ms 脉冲
    signal[beat_start:end] = np.random.randn(end - beat_start) * 0.8

signal = signal / (np.abs(signal).max() + 1e-8)
print(f'信号长度: {len(signal)} 采样 ({DURATION}s @ {SR}Hz)')
print(f'期望 BPM: 120，期望节拍周期: {beat_period} 采样 = {beat_period/SR*1000:.0f}ms')

## ✏️ 任务 1：实现 onset_envelope

In [ ]:
def my_onset_envelope(signal, sample_rate, n_fft=N_FFT, hop_length=HOP):
    """谱通量 onset 包络：相邻帧幅度谱正差值之和。"""
    # ✏️ TODO：调用 stft，取绝对值，计算差分，半波整流，按频率求和
    raise NotImplementedError("TODO")

env = my_onset_envelope(signal, SR)
print(f'onset envelope shape: {env.shape}')
fps = SR / HOP
print(f'帧率: {fps:.1f} fps')

plt.figure(figsize=(10, 2))
times = np.arange(len(env)) / fps
plt.plot(times, env)
plt.xlabel('时间 (s)')
plt.title('Onset Envelope（谱通量）')
plt.tight_layout()
plt.show()

## ✏️ 任务 2：用自相关估计 BPM

In [ ]:
def my_beat_track(signal, sample_rate, hop_length=HOP):
    """从自相关估计 BPM 和节拍时间。"""
    env = my_onset_envelope(signal, sample_rate, hop_length=hop_length)
    fps = sample_rate / hop_length

    # BPM 范围 40-240 对应的 lag 范围（单位：帧）
    min_lag = max(1, int(fps * 60 / 240))
    max_lag = min(len(env) - 1, int(fps * 60 / 40))
    lags = np.arange(min_lag, max_lag + 1)

    # ✏️ TODO: 对每个 lag 计算自相关（env 与移位 env 的点积）
    raise NotImplementedError("TODO")

bpm, beats = my_beat_track(signal, SR)
print(f'估计 BPM: {bpm:.1f}  (期望: 120)')
print(f'节拍数: {len(beats)}  (期望: {int(DURATION * 2)})')
assert abs(bpm - 120) < 15, f'BPM 偏差过大: {bpm:.1f}'
print('✅ 节拍追踪验证通过')

In [ ]:
# 与 aurora.music 对比
from aurora.music import onset_envelope, beat_track

ref_env = onset_envelope(signal, SR, hop_length=HOP)
ref_bpm, ref_beats = beat_track(signal, SR, hop_length=HOP)

print(f'自实现 BPM:      {bpm:.1f}')
print(f'aurora.music BPM: {ref_bpm:.1f}')
print(f'误差: {abs(bpm - ref_bpm):.1f} BPM')

## 本课收束

| 概念 | 要记住的 |
|---|---|
| 谱通量 | 正差值之和，检测能量突增 |
| 自相关 | lag=τ 处最大值 → 周期 τ → BPM |
| aurora.music | beat_track() 是你刚实现的算法的参考实现 |
| L79 | 有了 chroma + RMS + BPM 特征，就可以构建音乐嵌入向量 |

下一课（L79）将用对比学习（contrastive learning）训练音乐嵌入：三元组损失（triplet loss）让同类曲目在向量空间中靠近、不同类拉远。